# Unit 1: Pandas Fundamentals & The NumPy Bridge 🟢

**Estimated Time:** 2.5 hours  
**Prerequisites:** Phase 1 NumPy complete, basic understanding of financial returns  
**Equipment:** VSCode, Jupyter notebook, `pandas` and `numpy` installed in your `.venv`

---

## Why This Matters in Quant Finance 🎯

**The Alignment Problem:**

You're building a multi-asset portfolio tracker. You have:
- **AAPL**: price data for every trading day in 2024
- **TSLA**: missing data for 15 days (halts, your data vendor had gaps)
- **MSFT**: started tracking mid-year, only have H2 data

You need to compute daily returns, correlation matrices, and portfolio variance. With pure NumPy, you'd spend hours writing alignment logic, handling missing dates, and debugging shape mismatches. **One indexing error and your Sharpe ratio is garbage.**

Pandas solves this: it carries **labels** (dates, ticker symbols) alongside your data and **automatically aligns** operations. When you compute `aapl_returns - tsla_returns`, Pandas matches dates for you. When dates don't overlap, it handles the NaNs gracefully.

**But here's the trap:** automatic alignment can also **hide bugs**. If your AAPL data is daily and your TSLA data is weekly, Pandas will silently forward-fill or leave NaNs, and you won't notice until your backtest returns look suspiciously good.

**This unit teaches you:**
- How Pandas extends NumPy with labels (the "bridge")
- When automatic alignment saves you vs when it silently breaks your code
- When to stay in Pandas vs when to drop down to NumPy for speed
- How to write robust return calculations that handle real-world messy data

---

## Learning Objectives 📚

By the end of this unit, you will be able to:

1. **Explain the relationship** between Pandas and NumPy using the "NumPy with labels" mental model
2. **Predict alignment behavior** when operating on Series/DataFrames with different indexes
3. **Choose appropriately** between staying in Pandas vs converting to NumPy for performance
4. **Debug alignment issues** by inspecting indexes and using `.reindex()` or explicit joins
5. **Build return calculators** that handle misaligned timestamps and missing data correctly
6. **Measure and understand** the performance cost of labeled operations vs raw NumPy

---

## Core Concepts — Expanded 🔍

### 1. Series = 1D NumPy Array + Index

**The NumPy version:**
```python
import numpy as np

prices = np.array([150.0, 152.3, 151.8, 153.2])
# If you want to know the price on day 2: prices[2]
# But what DATE is day 2? You need a separate variable.
```

**The Pandas version:**
```python
import pandas as pd

prices = pd.Series(
    [150.0, 152.3, 151.8, 153.2],
    index=['2024-01-02', '2024-01-03', '2024-01-04', '2024-01-05']
)
# Now: prices['2024-01-03'] → 152.3
# The date is ATTACHED to the data, not separate.
```

**Key insight:** The Index is **metadata that travels with the data**. When you slice, filter, or compute, the Index comes along. This prevents the classic "off-by-one indexing error" that destroys quant analyses.

**Quant finance use case:** Computing returns over weekends. With NumPy, you'd manually track that "Monday price - Friday price" isn't a 1-day return. With Pandas + DatetimeIndex, you can compute `(prices.pct_change() * trading_days)` and the Index tells you exactly how many days elapsed.

---

### 2. DataFrame = 2D NumPy Array + Row Index + Column Index

**The NumPy version:**
```python
# Portfolio: 3 stocks, 4 days
portfolio = np.array([
    [150.0, 2800.0, 180.0],  # day 0: AAPL, TSLA, MSFT
    [152.3, 2750.0, 182.1],  # day 1
    [151.8, 2820.0, 181.5],  # day 2
    [153.2, 2790.0, 183.0]   # day 3
])
# To get TSLA day 1: portfolio[1, 1]
# But which column is TSLA? You need to remember "column 1 = TSLA"
```

**The Pandas version:**
```python
portfolio = pd.DataFrame(
    [[150.0, 2800.0, 180.0],
     [152.3, 2750.0, 182.1],
     [151.8, 2820.0, 181.5],
     [153.2, 2790.0, 183.0]],
    index=['2024-01-02', '2024-01-03', '2024-01-04', '2024-01-05'],
    columns=['AAPL', 'TSLA', 'MSFT']
)
# Now: portfolio.loc['2024-01-03', 'TSLA'] → 2750.0
# Both row (date) AND column (ticker) are labeled.
```

**Key insight:** DataFrames have **dual indexing** — rows have an Index, columns have an Index. This means operations can align on **both dimensions** simultaneously.

**Quant finance use case:** Correlation matrices. With NumPy, if you accidentally reorder your columns, your correlation matrix silently becomes wrong (AAPL correlated with MSFT's data). With Pandas, column labels prevent this — the correlation matrix knows which ticker is which.

---

### 3. Automatic Alignment — The Blessing and the Curse

**Scenario:** You have returns for two stocks with different trading days.

```python
# AAPL traded all 4 days
aapl = pd.Series([0.01, 0.02, -0.01, 0.015], 
                 index=['2024-01-02', '2024-01-03', '2024-01-04', '2024-01-05'])

# TSLA was halted on Jan 3rd (missing data)
tsla = pd.Series([0.015, -0.008, 0.012], 
                 index=['2024-01-02', '2024-01-04', '2024-01-05'])

# What happens when you compute the spread?
spread = aapl - tsla
```

**Pandas automatically aligns on the Index:**
```
2024-01-02   -0.005   # both exist: 0.01 - 0.015
2024-01-03    NaN     # only AAPL exists: 0.02 - ?
2024-01-04   -0.002   # both exist: -0.01 - (-0.008)
2024-01-05    0.003   # both exist: 0.015 - 0.012
```

**The blessing:** You didn't have to write alignment code. Pandas matched the dates for you.

**The curse:** If you weren't expecting that NaN, and you later compute `spread.mean()`, Pandas **silently ignores NaN** by default. Your mean is computed over 3 days instead of 4, and you might not notice.

**NumPy equivalent (for comparison):**
```python
# With NumPy, you'd have to:
# 1. Manually create a common date index
# 2. Reindex both arrays to that common index
# 3. Fill missing values with NaN (or your chosen strategy)
# 4. Then subtract
# Error-prone and verbose!
```

**Key principle:** **Alignment happens on the Index, not position**. This is fundamentally different from NumPy broadcasting, which aligns on **shape and position**.

---

### 4. When to Drop to NumPy: `.values` vs `.to_numpy()` vs Staying in Pandas

**The performance question:** Pandas operations are slower than pure NumPy because of:
1. **Index lookups** (hash table overhead)
2. **Alignment checks** (comparing indexes before operating)
3. **Object dtype overhead** (some Pandas columns aren't pure numeric arrays)

**Rule of thumb:**
- **Stay in Pandas** for: anything involving alignment (time series with different dates), joining datasets, grouping/aggregating, reading/writing files
- **Drop to NumPy** for: tight loops, element-wise math on aligned data, performance-critical inner loops (e.g., Monte Carlo simulations running millions of paths)

**How to convert:**

```python
# Pandas Series → NumPy array
returns = pd.Series([0.01, 0.02, -0.01], index=['2024-01-02', '2024-01-03', '2024-01-04'])

# Option 1: .values (legacy, discouraged)
returns.values  # → array([0.01, 0.02, -0.01])
# Problem: returns.values on mixed-type data can give you an OBJECT array (slow!)

# Option 2: .to_numpy() (preferred)
returns.to_numpy()  # → array([0.01, 0.02, -0.01])
# Safer: always returns a proper NumPy array, with controllable dtype

# Option 3: Stay in Pandas but access underlying array
returns.array  # → PandasArray, not NumPy (Pandas extension array)
```

**When you convert to NumPy, you LOSE the Index.** This is intentional for performance, but it means **you must be certain your data is already aligned** before dropping to NumPy.

**Quant finance example:**
```python
# Computing cumulative returns: Pandas is fine (alignment matters)
cum_returns = (1 + returns).cumprod() - 1

# Monte Carlo simulation: drop to NumPy for speed (no alignment needed inside the loop)
np_returns = returns.to_numpy()
paths = np.random.normal(np_returns.mean(), np_returns.std(), size=(10000, 252))
```

---

### 5. Automatic Alignment vs NumPy Broadcasting — The Key Difference

**NumPy broadcasting:** aligns on **shape**, starting from the **trailing (rightmost) dimensions**.

```python
a = np.array([1, 2, 3])        # shape (3,)
b = np.array([[10], [20]])     # shape (2, 1)

a + b  # Broadcasting: (2,1) + (3,) → (2,3)
# Result:
# [[11, 12, 13],
#  [21, 22, 23]]
```

**Pandas alignment:** aligns on **Index labels**, **regardless of shape or position**.

```python
s1 = pd.Series([1, 2, 3], index=['a', 'b', 'c'])
s2 = pd.Series([10, 20], index=['b', 'c'])

s1 + s2
# Alignment on labels:
# a    NaN   (s1 has 'a', s2 doesn't)
# b    12    (both have 'b': 2 + 10)
# c    23    (both have 'c': 3 + 20)
```

**This difference matters when:** you're working with time series that have gaps (trading halts, weekends, different start dates). NumPy broadcasting would compute nonsense; Pandas alignment does the right thing.

---

## Hands-On Tasks 💻

**Setup:**
```bash
# Activate your virtual environment
cd ~/Prometheus
source .venv/bin/activate

# Create a new notebook for this unit
jupyter notebook
# Create: Unit_1_Pandas_NumPy_Bridge.ipynb
```

**Import setup cell:**
```python
import numpy as np
import pandas as pd
import time

# Set display options for cleaner output
pd.set_option('display.precision', 6)
pd.set_option('display.max_rows', 20)

print(f"NumPy version: {np.__version__}")
print(f"Pandas version: {pd.__version__}")
```

---

### Task 1: Aligned vs Misaligned Series Operations (30 min)

**Objective:** Understand what happens when Series indexes match vs don't match.

```python
# Create two Series with FULL overlap
aapl_aligned = pd.Series(
    [150.0, 152.3, 151.8, 153.2],
    index=['2024-01-02', '2024-01-03', '2024-01-04', '2024-01-05'],
    name='AAPL'
)

msft_aligned = pd.Series(
    [180.0, 182.1, 181.5, 183.0],
    index=['2024-01-02', '2024-01-03', '2024-01-04', '2024-01-05'],
    name='MSFT'
)

# Create a Series with PARTIAL overlap
tsla_misaligned = pd.Series(
    [2800.0, 2820.0, 2790.0],
    index=['2024-01-02', '2024-01-04', '2024-01-05'],  # Missing Jan 3rd
    name='TSLA'
)

# TODO for you:
# 1. Compute: spread_aligned = aapl_aligned - msft_aligned
#    - Print the result
#    - Verify: no NaNs, all 4 dates present
#
# 2. Compute: spread_misaligned = aapl_aligned - tsla_misaligned
#    - Print the result
#    - Q: What happens on 2024-01-03? Why?
#
# 3. Compute the mean of both spreads
#    - Print both means
#    - Q: How does Pandas handle the NaN in spread_misaligned.mean()?
#
# 4. Explicitly check for NaNs:
#    - Use .isna() to see where NaNs appear
#    - Count NaNs using .isna().sum()
#
# 5. Create a COMPLETELY non-overlapping Series:
#    goog = pd.Series([2900, 2950], index=['2024-01-08', '2024-01-09'])
#    - Compute: aapl_aligned - goog
#    - Q: What does the result look like? What's the Index?
```

**Expected learning:**
- When indexes fully overlap → clean result, no NaNs
- When indexes partially overlap → NaNs where dates don't match
- When indexes don't overlap at all → **all NaNs**, but the Index is the **union** of both indexes
- `skipna=True` is the default for aggregation methods (`.mean()`, `.sum()`, etc.)

---

### Task 2: NumPy Broadcasting vs Pandas Alignment (30 min)

**Objective:** See how the same operation behaves differently in NumPy vs Pandas.

```python
# Same data, two formats
prices_pandas = pd.Series([150.0, 152.3, 151.8], index=[0, 1, 2])
prices_numpy = prices_pandas.to_numpy()

# A scalar adjustment
adjustment = 5.0

# TODO for you:
# 1. NumPy operation:
#    adjusted_np = prices_numpy + adjustment
#    - Print the result
#    - Q: What happened? (broadcasting of scalar to array)
#
# 2. Pandas operation:
#    adjusted_pd = prices_pandas + adjustment
#    - Print the result
#    - Q: Same mathematical result as NumPy? What about the Index?
#
# Now create a MORE INTERESTING case:
# A Series with a DIFFERENT Index
adjustment_series = pd.Series([5.0, 10.0], index=[1, 2])  # Only has index 1 and 2

# 3. Pandas alignment operation:
#    adjusted_aligned = prices_pandas + adjustment_series
#    - Print the result
#    - Q: What happens at index 0? Why?
#    - Q: What happens at index 1 and 2?
#
# 4. What if you WANTED NumPy broadcasting behavior in Pandas?
#    - Use .values to drop to NumPy for one side:
#    mixed = prices_pandas + adjustment_series.values
#    - Q: Does this work? What error/warning do you get? Why?
```

**Expected learning:**
- NumPy broadcasting: positional alignment based on shape
- Pandas alignment: label-based alignment, NaN for non-overlapping labels
- Mixing NumPy arrays and Pandas Series can cause **shape errors** or **silent bugs** (positional indexing sneaks back in)

---

### Task 3: Performance Comparison — Pure NumPy vs Pandas (45 min)

**Objective:** Measure the overhead of labeled operations and understand when to optimize.

```python
# Generate large datasets
n = 1_000_000

# NumPy arrays
np_array1 = np.random.randn(n)
np_array2 = np.random.randn(n)

# Pandas Series (with integer index)
pd_series1 = pd.Series(np_array1)
pd_series2 = pd.Series(np_array2)

# Pandas Series (with DatetimeIndex - more realistic for finance)
dates = pd.date_range('2020-01-01', periods=n, freq='min')  # 1 million minutes
pd_series_dt1 = pd.Series(np_array1, index=dates)
pd_series_dt2 = pd.Series(np_array2, index=dates)

# TODO for you:
# 1. Benchmark element-wise addition with NumPy:
#    %%timeit
#    result_np = np_array1 + np_array2
#
# 2. Benchmark element-wise addition with Pandas (integer index):
#    %%timeit
#    result_pd = pd_series1 + pd_series2
#
# 3. Benchmark element-wise addition with Pandas (DatetimeIndex):
#    %%timeit
#    result_pd_dt = pd_series_dt1 + pd_series_dt2
#
# 4. Benchmark a more complex operation (moving average):
#    - NumPy version (manual convolution):
#      window = 20
#      %%timeit
#      np_ma = np.convolve(np_array1, np.ones(window)/window, mode='valid')
#
#    - Pandas version (built-in rolling):
#      %%timeit
#      pd_ma = pd_series_dt1.rolling(window).mean()
#
# 5. Record your results in a markdown cell:
#    Create a simple table showing:
#    Operation | NumPy Time | Pandas (int index) | Pandas (DatetimeIndex)
#
# 6. Analysis questions:
#    - What's the overhead ratio? (Pandas time / NumPy time)
#    - Does the index type (integer vs datetime) matter for performance?
#    - For which operation is the overhead largest?
```

**Expected learning:**
- Pandas has measurable overhead (typically 2-5x slower for element-wise ops)
- DatetimeIndex is slower than integer index (more complex hashing/comparison)
- For built-in operations (rolling, groupby), Pandas is highly optimized and the overhead is worth it
- For tight loops with millions of iterations, drop to NumPy

**Performance insight for quant finance:**
- Backtesting a strategy over 10 years of minute data → ~2.6M data points
- If your strategy recomputes indicators every bar: overhead compounds
- Strategy: use Pandas for data alignment and I/O, drop to NumPy for the hot loop

---

### Task 4: Build a Misaligned Return Calculator (45 min)

**Objective:** Handle real-world messy data — different start dates, trading halts, missing values.

```python
# Simulate realistic stock data with issues
np.random.seed(42)

# AAPL: full data, 252 trading days
dates_aapl = pd.bdate_range('2024-01-01', periods=252)
prices_aapl = 150 * (1 + np.random.randn(252).cumsum() * 0.01)
aapl = pd.Series(prices_aapl, index=dates_aapl, name='AAPL')

# TSLA: started tracking late, only 200 days
dates_tsla = pd.bdate_range('2024-03-15', periods=200)
prices_tsla = 2800 * (1 + np.random.randn(200).cumsum() * 0.015)
tsla = pd.Series(prices_tsla, index=dates_tsla, name='TSLA')

# MSFT: has 10 random missing days (trading halts, data gaps)
dates_msft = pd.bdate_range('2024-01-01', periods=252)
missing_indices = np.random.choice(252, size=10, replace=False)
dates_msft_filtered = dates_msft.delete(missing_indices)
prices_msft = 180 * (1 + np.random.randn(len(dates_msft_filtered)).cumsum() * 0.008)
msft = pd.Series(prices_msft, index=dates_msft_filtered, name='MSFT')

# TODO for you:
# 1. Compute simple returns for each stock:
#    - Formula: (price_t - price_t-1) / price_t-1
#    - Use .pct_change()
#    - Store as: aapl_ret, tsla_ret, msft_ret
#
# 2. Inspect the returns:
#    - Print: aapl_ret.head(), aapl_ret.tail()
#    - Q: What's the first value in each return series? Why is it NaN?
#
# 3. Combine all three into a DataFrame:
#    returns_df = pd.DataFrame({
#        'AAPL': aapl_ret,
#        'TSLA': tsla_ret,
#        'MSFT': msft_ret
#    })
#    - Print: returns_df.head(10)
#    - Q: What dates appear in the DataFrame? (Union of all indexes?)
#    - Q: Where do you see NaNs? Why?
#
# 4. Count missing values per stock:
#    - Use: returns_df.isna().sum()
#    - Q: Which stock has the most NaNs? Why?
#
# 5. Compute correlation matrix:
#    - Use: returns_df.corr()
#    - Q: How does Pandas handle NaNs when computing correlations?
#    - Try: returns_df.corr(min_periods=50)
#    - Q: What does min_periods do?
#
# 6. Write a function to compute LOG returns instead:
def compute_log_returns(prices_series):
    """
    Compute log returns: ln(P_t / P_t-1)
    
    Args:
        prices_series: pd.Series with prices, Index must be dates
    
    Returns:
        pd.Series with log returns, same Index as input (first value NaN)
    """
    # TODO: Implement this
    # Hint: Use np.log() and Series.shift()
    pass

#    - Test your function on all three stocks
#    - Verify: first value is NaN, Index is preserved
#
# 7. BONUS: Alignment behavior documentation
#    - Create a small example showing what happens when you compute:
#      portfolio_return = 0.5 * aapl_ret + 0.3 * tsla_ret + 0.2 * msft_ret
#    - Q: On dates where only AAPL has data, what's the portfolio return?
#    - Q: Is this the behavior you want? If not, how would you fix it?
#      (Hint: .dropna() or .fillna() or explicit reindexing)
```

**Expected learning:**
- `.pct_change()` automatically handles alignment (compares consecutive index values, not positions)
- Combining Series with different indexes into a DataFrame → **outer join** by default (union of indexes, NaNs fill gaps)
- Correlation with NaNs: Pandas uses pairwise-complete observations by default
- Log returns vs simple returns: same alignment behavior, different numerical stability properties (connects to your Phase 1 weak spot!)

---

## Self-Check Questions 🧠

Answer these **before looking at hints**. Write your answers in a markdown cell in your notebook.

### Q1: Index Alignment Mechanics
You have two Series:
```python
s1 = pd.Series([10, 20, 30], index=['a', 'b', 'c'])
s2 = pd.Series([1, 2], index=['b', 'c'])
```

What is the result of `s1 + s2`? Write out the full result (index and values).

<details>
<summary>Hint 1</summary>
Pandas aligns on labels, not positions. What labels do both Series share?
</details>

<details>
<summary>Hint 2</summary>
What happens at index 'a', where only s1 has a value?
</details>

---

### Q2: When to Use `.to_numpy()`
Your colleague writes:
```python
returns = portfolio_df['AAPL']  # A Series
mean_return = returns.values.mean()
```

Is this code correct? If not, why not? If it works, is it **good practice**?

<details>
<summary>Hint 1</summary>
What type does `.values` return? Is it always a NumPy array?
</details>

<details>
<summary>Hint 2</summary>
What's the difference between `.values` and `.to_numpy()`?
</details>

---

### Q3: Performance Cost Understanding
You need to compute daily returns for a DataFrame with 1000 stocks and 5 years of data (~1260 rows × 1000 columns). You have two options:

**Option A:** Stay in Pandas and use `df.pct_change()`  
**Option B:** Convert to NumPy, compute manually, then rebuild the DataFrame

Which is faster? Why? When would you choose the slower option anyway?

<details>
<summary>Hint 1</summary>
Pandas operations on DataFrames are vectorized and highly optimized. What's the overhead?
</details>

<details>
<summary>Hint 2</summary>
Consider: alignment checks, index preservation, and built-in NaN handling. Are those worth the overhead?
</details>

---

### Q4: Alignment vs Broadcasting
Explain the difference between these two operations:

```python
# Operation 1
s = pd.Series([1, 2, 3], index=['a', 'b', 'c'])
result1 = s + 10

# Operation 2
s1 = pd.Series([1, 2, 3], index=['a', 'b', 'c'])
s2 = pd.Series([10, 20], index=['a', 'b'])
result2 = s1 + s2
```

In which case does Pandas use "alignment"? In which case does it use "broadcasting"? What's the difference?

<details>
<summary>Hint</summary>
When one operand is a scalar, Pandas broadcasts (like NumPy). When both operands have indexes, Pandas aligns on labels.
</details>

---

### Q5: Real-World Debugging Scenario
Your backtest returns look too good. You discover that your DataFrame has:
- **Returns for Stock A:** daily data from Jan 1 to Dec 31
- **Returns for Stock B:** weekly data (only Fridays) from Jan 1 to Dec 31

You computed portfolio returns as:
```python
portfolio = 0.5 * df['Stock_A'] + 0.5 * df['Stock_B']
```

What went wrong? How does automatic alignment create a bug here?

<details>
<summary>Hint 1</summary>
What happens on non-Friday days? Does Stock B have values for those dates?
</details>

<details>
<summary>Hint 2</summary>
When one stock has NaN on a given day, what's the portfolio return? Is it 50% of Stock A's return (because the weight on NaN is lost), or is it NaN (because you can't compute 50% + NaN)?
</details>

---

## End of Unit Deliverable 🎯

**Build:** A Python function that computes log returns for a dictionary of stock price Series with misaligned dates. The function must document alignment behavior and handle edge cases.

### Specifications:

```python
def compute_multi_stock_log_returns(price_dict, min_overlap=20):
    """
    Compute log returns for multiple stocks with different trading calendars.
    
    Parameters:
    -----------
    price_dict : dict
        Dictionary mapping stock tickers (str) to price Series (pd.Series).
        Each Series must have a DatetimeIndex.
        Series may have different lengths and different date ranges.
    
    min_overlap : int, default=20
        Minimum number of overlapping dates required between any two stocks
        to include them in the output. Stocks with fewer than min_overlap
        common dates are excluded (with a warning).
    
    Returns:
    --------
    returns_df : pd.DataFrame
        DataFrame with log returns for all stocks.
        Index: union of all dates from all stocks (outer join)
        Columns: stock tickers
        Values: log returns (NaN where stock didn't trade)
    
    alignment_report : dict
        Dictionary containing:
        - 'total_dates': int, total unique dates across all stocks
        - 'common_dates': int, dates where ALL stocks have data
        - 'per_stock_coverage': dict mapping ticker -> % of dates covered
        - 'pairwise_overlap': pd.DataFrame, matrix of pairwise overlap counts
    
    Raises:
    -------
    ValueError
        If any price Series has a non-DatetimeIndex
        If any price Series has fewer than 2 data points
    
    Example:
    --------
    >>> prices = {
    ...     'AAPL': pd.Series([150, 152, 151], 
    ...                       index=pd.date_range('2024-01-01', periods=3)),
    ...     'TSLA': pd.Series([2800, 2750], 
    ...                       index=pd.date_range('2024-01-02', periods=2))
    ... }
    >>> returns, report = compute_multi_stock_log_returns(prices)
    >>> print(returns)
    # Should show:
    #             AAPL      TSLA
    # 2024-01-01   NaN       NaN
    # 2024-01-02  0.0132      NaN
    # 2024-01-03 -0.0066  -0.0179
    
    Notes:
    ------
    - Log returns are computed as ln(P_t / P_{t-1})
    - First return for each stock is NaN (no previous price)
    - Alignment is automatic: dates don't need to match across stocks
    - Use min_overlap to filter out stocks with insufficient overlap
    """
    # TODO: Implement this function
    pass
```

### What to submit (in your notebook):

1. **Function implementation** (code cell)
2. **Test cases** demonstrating:
   - Fully aligned stocks (same dates)
   - Partially aligned stocks (different date ranges)
   - Stocks with missing dates (gaps in the middle)
   - Edge case: stock with only 1 price (should raise ValueError)
3. **Alignment report interpretation** (markdown cell):
   - Explain what each field in `alignment_report` tells you
   - Show an example report and interpret the results
4. **Numerical validation** (code cell):
   - Pick two dates where both stocks have data
   - Manually compute the log return for one stock using `np.log()`
   - Verify it matches your function's output
5. **Performance note** (markdown cell):
   - Would you convert to NumPy inside this function? Why or why not?
   - Where is the performance bottleneck likely to be?

### Grading yourself (self-assessment):

- ✅ **Pass:** Function handles all test cases, produces correct returns, alignment report is accurate
- 🌟 **Strong:** Function also includes helpful warnings (e.g., "TSLA and GOOG have only 5 overlapping dates, below min_overlap threshold"), efficient implementation, clear docstring
- 🚀 **Exceptional:** Function additionally handles timezone-aware DatetimeIndexes, provides visualization of overlap matrix (heatmap), includes option to forward-fill NaNs or use business day calendar

---

## Exit Criteria — You're Ready for Unit 2 When: ✅

Self-certify that you can do ALL of the following **without looking at notes**:

1. [ ] **Explain to a colleague:** "Pandas is NumPy with labels. Here's what that means..." (use a concrete example)
2. [ ] **Predict behavior:** Given two Series with partial index overlap, write out the result of `s1 + s2` on paper before running the code
3. [ ] **Debug alignment:** When your correlation matrix looks wrong, check `df.index` and `df.columns` to spot misalignment
4. [ ] **Make the performance trade-off:** Justify staying in Pandas vs converting to NumPy for a specific operation
5. [ ] **Write robust code:** Create a function that handles misaligned time series and includes alignment checks
6. [ ] **Recognize the trap:** Spot when automatic alignment is hiding a bug (e.g., weekly data mixed with daily data)

**Confidence check:** Re-run your Unit 1 deliverable with NEW random stock data (different seed). Does it still work correctly? Does the alignment report make sense?